# Random Forests implementation from scratch

### Imports

Loads the core libraries needed for the whole notebook — **pandas** for tabular data handling, **numpy** for the vectorized math behind the model, **Counter** for majority voting, **train_test_split** for training, and **load_breast_cancer** from scikit-learn as a real-data source.

In [1]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

### Loading Data

Loads the real Breast Cancer Wisconsin dataset, using all 30 real tumor measurements as features and converting the numeric diagnosis code into readable labels (malignant/benign). Splits the data into training and test sets so the model can be evaluated on patients it never saw during training.

In [2]:
data = load_breast_cancer(as_frame=True)
df = data.frame

X = df.drop(columns=["target"]).values.astype(float)
y = np.array(["malignant" if t == 0 else "benign" for t in df["target"].values])
feature_names = list(df.drop(columns=["target"]).columns)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

print(f"Training examples: {X_train.shape[0]}, Test examples: {X_test.shape[0]}, Features: {X_train.shape[1]}")

Training examples: 455, Test examples: 114, Features: 30


### Node Class

Defines the building block of a decision tree. Each **Node** is either a decision node holding a feature index, a threshold to split on, and pointers to a left/right child or a leaf node, holding a final predicted class with no children.

In [3]:
class Node:
    def __init__(self, feature_idx=None, threshold=None, info_gain=None, left=None, right=None, value=None):
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.info_gain = info_gain
        self.left = left
        self.right = right
        self.value = value

### Decision Tree Class

Implements a single decision tree from scratch: recursively finds the feature and threshold that best separates the two classes by maximizing information gain (entropy reduction), splits the data accordingly, and repeats on each half until hitting **max_depth** or running out of samples. Includes an **n_features parameter**, meaning when set, each split only considers a random subset of the 30 features rather than all of them, which is what makes this a building block for a random forest specifically, rather than a plain decision tree: restricting each split's choices forces different trees to develop differently instead of every tree converging on the same strongest feature.


In [4]:
class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=5, n_features=None):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.n_features = n_features

    def build_tree(self, X, y, curr_depth=0):
        n_samples, n_total_features = X.shape

        if n_samples >= self.min_samples_split and curr_depth <= self.max_depth:
            feature_indices = (np.random.choice(n_total_features, self.n_features, replace=False)
                                if self.n_features else range(n_total_features))
            best_split = self.best_split(X, y, feature_indices)

            if best_split["info_gain"] > 0:
                left_node = self.build_tree(best_split["X_left"], best_split["y_left"], curr_depth + 1)
                right_node = self.build_tree(best_split["X_right"], best_split["y_right"], curr_depth + 1)
                return Node(best_split["feature_idx"], best_split["threshold"],
                            best_split["info_gain"], left_node, right_node)

        leaf_value = Counter(y).most_common(1)[0][0]
        return Node(value=leaf_value)

    def best_split(self, X, y, feature_indices):
        best = {'feature_idx': None, 'threshold': None, 'info_gain': -1,
                'X_left': None, 'y_left': None, 'X_right': None, 'y_right': None}

        for feature_idx in feature_indices:
            thresholds = np.unique(X[:, feature_idx])
            for threshold in thresholds:
                mask = X[:, feature_idx] <= threshold
                X_left, y_left = X[mask], y[mask]
                X_right, y_right = X[~mask], y[~mask]

                if len(y_left) and len(y_right):
                    info_gain = self.information_gain(y, y_left, y_right)
                    if info_gain > best['info_gain']:
                        best.update(feature_idx=feature_idx, threshold=threshold, info_gain=info_gain,
                                    X_left=X_left, y_left=y_left, X_right=X_right, y_right=y_right)
        return best

    def information_gain(self, parent_y, left_y, right_y):
        left_weight = len(left_y) / len(parent_y)
        right_weight = len(right_y) / len(parent_y)
        return self.entropy(parent_y) - (left_weight * self.entropy(left_y) + right_weight * self.entropy(right_y))

    def entropy(self, y):
        entropy = 0
        for class_label in np.unique(y):
            p = len(y[y == class_label]) / len(y)
            entropy += -p * np.log2(p)
        return entropy

    def fit(self, X, y):
        self.root = self.build_tree(X, y)

    def predict(self, X):
        return np.array([self.predict_class(row, self.root) for row in X])

    def predict_class(self, row, node):
        if node.value is not None:
            return node.value
        if row[node.feature_idx] <= node.threshold:
            return self.predict_class(row, node.left)
        else:
            return self.predict_class(row, node.right)

### Random Forest Class

Builds an ensemble of many decision trees and combines their votes. **bootstrap_sample** draws a random sample of patients with replacement for each tree, so every tree trains on a slightly different, overlapping view of the real training data. **fit** builds **n_trees** such trees, each also restricted to a random subset of features per split (√30 ≈ 5–6, by default). **predict** runs every tree on new data and returns whichever class the majority of trees voted for.

In [5]:
class RandomForest:
    def __init__(self, n_trees=20, max_depth=5, min_samples_split=2, n_features=None):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_features = n_features
        self.trees = []

    def bootstrap_sample(self, X, y):
        n_samples = X.shape[0]
        indices = np.random.choice(n_samples, n_samples, replace=True)
        return X[indices], y[indices]

    def fit(self, X, y):
        self.trees = []
        n_features_per_tree = self.n_features or int(np.sqrt(X.shape[1]))

        for _ in range(self.n_trees):
            tree = DecisionTree(min_samples_split=self.min_samples_split,
                                 max_depth=self.max_depth, n_features=n_features_per_tree)
            X_sample, y_sample = self.bootstrap_sample(X, y)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        tree_predictions = np.array([tree.predict(X) for tree in self.trees])
        predictions = []
        for i in range(X.shape[0]):
            votes = tree_predictions[:, i]
            predictions.append(Counter(votes).most_common(1)[0][0])
        return np.array(predictions)

### Training & Predicting

Trains a forest of 20 trees (each up to depth 5) on the real training data, then predicts on the 114 held-out test patients and measures accuracy.

In [6]:
np.random.seed(2)

rf = RandomForest(n_trees=20, max_depth=5)
rf.fit(X_train, y_train)

predictions = rf.predict(X_test)
accuracy = np.mean(predictions == y_test) * 100
print(f"Test accuracy: {accuracy:.2f}%")

Test accuracy: 94.74%


**94.74% test accuracy** meaning that 108 of 114 real, previously unseen patients were correctly classified.

### Feature Importance

Sums the information gain contributed by each feature every time it's used to split across all 20 trees, giving a measure of which real tumor measurements the forest as a whole relies on most.

In [7]:
from collections import defaultdict

importance = defaultdict(float)

def accumulate_importance(node):
    if node.value is not None:
        return
    importance[node.feature_idx] += node.info_gain
    accumulate_importance(node.left)
    accumulate_importance(node.right)

for tree in rf.trees:
    accumulate_importance(tree.root)

total_importance = sum(importance.values())
sorted_importance = sorted(importance.items(), key=lambda x: -x[1])

for idx, val in sorted_importance[:10]:
    print(f"{feature_names[idx]}: {val/total_importance*100:.2f}%")

worst perimeter: 6.49%
worst concave points: 6.43%
mean smoothness: 5.67%
worst area: 5.24%
worst smoothness: 5.04%
worst radius: 5.02%
perimeter error: 4.80%
worst concavity: 4.38%
fractal dimension error: 4.31%
mean perimeter: 4.27%


**worst perimeter** (6.49%) and **worst concave points** (6.43%) rank highest, meaning the most extreme (worst) measurements of tumor size and shape irregularity found in each biopsy image are the strongest real predictors of malignancy in this dataset.

### Sklearn Validation

Fits scikit-learn's **RandomForestClassifier** with matching hyperparameters (**n_estimators=20**, **max_depth=5**, **criterion='entropy'**) on the identical data, as an independent correctness check on the from-scratch implementation.

In [8]:
from sklearn.ensemble import RandomForestClassifier

sk_rf = RandomForestClassifier(n_estimators=20, max_depth=5, criterion='entropy', random_state=2)
sk_rf.fit(X_train, y_train)
sk_accuracy = sk_rf.score(X_test, y_test) * 100

print(f"From scratch: {accuracy:.2f}%")
print(f"sklearn:      {sk_accuracy:.2f}%")

From scratch: 94.74%
sklearn:      94.74%


94.74% for both. 